In [ ]:
!pip install -q kaggle-environments huggingface_hub

In [ ]:
import os
from getpass import getpass

if not os.environ.get('HF_TOKEN'):
    token = getpass('HF_TOKEN: ').strip()
    if not token:
        raise RuntimeError('HF_TOKEN required to upload dataset to Hugging Face')
    os.environ['HF_TOKEN'] = token

print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import subprocess

# Clone the repo to get main.py + generate_training_data.py at the latest commit.
# If already cloned, pull instead.
import os
repo_dir = 'orbit-wars-agent'
if not os.path.exists(repo_dir):
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/DevaanshPathak/orbit-wars-agent.git', repo_dir],
        check=True,
    )
    print('Cloned repo.')
else:
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)
    print('Pulled latest.')

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────────────────────────
GAMES                  = 5000   # number of unique game seeds
WORKERS                = 4      # Kaggle CPU notebooks have 4 cores
MAX_CANDIDATES_PER_TURN = 64
HF_REPO_ID             = 'devaanshpa/orbit-wars-agent'

# NOTE: GPU is NOT used here — orbit wars simulation is pure Python CPU work.
# Run this on a Kaggle CPU notebook to save GPU quota for training.
# ──────────────────────────────────────────────────────────────────────────────

print(f'Games: {GAMES}  Workers: {WORKERS}  Candidates/turn: {MAX_CANDIDATES_PER_TURN}')
print(f'Total game-sides: {GAMES * 2} (both-sides enabled)')

In [ ]:
import subprocess, sys, os

env = dict(os.environ)  # passes HF_TOKEN into the subprocess

cmd = [
    sys.executable, '-u', 'generate_training_data.py',
    '--games',                  str(GAMES),
    '--both-sides',
    '--workers',                str(WORKERS),
    '--max-candidates-per-turn', str(MAX_CANDIDATES_PER_TURN),
    '--hf-repo-id',             HF_REPO_ID,
]

print('Running:', ' '.join(cmd))
print('-' * 60, flush=True)

proc = subprocess.Popen(
    cmd,
    cwd='orbit-wars-agent',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode not in (0, None):
    raise RuntimeError(f'Data generation exited with code {proc.returncode}')
print('Done.')